<a href="https://colab.research.google.com/github/Dania-Yasir/flyrak-project/blob/main/work/notebooks/model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##The Model Training


# ML-08 — Capstone Modeling Lane

This notebook trains and compares Logistic Regression, Decision Tree, and Random Forest against the Week 4 rule-based baseline.

The same data definition, feature window, outcome window, and evaluation target from Week 4 are kept so the comparison remains consistent.

In [1]:
%pip install -q duckdb huggingface_hub pandas numpy scikit-learn matplotlib

In [2]:
from google.colab import userdata

import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError(
        "HF_TOKEN was not found in Colab Secrets."
    )

con = duckdb.connect()

safe_token = HF_TOKEN.replace("'", "''")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{safe_token}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_MARCH = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

print("Connected successfully.")

Connected successfully.


In [3]:
schema = con.execute(
    f"DESCRIBE SELECT * FROM {FACT_MARCH}"
).df()

display(schema)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [4]:
date_col = "report_date"
client_col = "client_hash_id"
content_col = "content_hash_id"

impressions_col = "gsc_impressions"
clicks_col = "gsc_clicks"

sessions_col = "ga4_sessions"
engaged_sessions_col = "ga4_engaged_sessions"

ga4_available_col = "ga4_data_available"

print("Columns selected successfully.")

Columns selected successfully.


In [5]:
query = f"""
SELECT
    {client_col} AS client_id,
    {content_col} AS content_id,

    -- March 1-15: information available before prediction
    SUM(
        CASE
            WHEN {date_col} BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
            THEN COALESCE({impressions_col}, 0)
            ELSE 0
        END
    ) AS imp_first_half,

    SUM(
        CASE
            WHEN {date_col} BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
            THEN COALESCE({clicks_col}, 0)
            ELSE 0
        END
    ) AS clicks_first_half,

    SUM(
        CASE
            WHEN {date_col} BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                 AND COALESCE({ga4_available_col}, FALSE)
            THEN COALESCE({sessions_col}, 0)
            ELSE 0
        END
    ) AS sessions_first_half,

    SUM(
        CASE
            WHEN {date_col} BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                 AND COALESCE({ga4_available_col}, FALSE)
            THEN COALESCE({engaged_sessions_col}, 0)
            ELSE 0
        END
    ) AS engaged_sessions_first_half,

    SUM(
        CASE
            WHEN {date_col} BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                 AND COALESCE({ga4_available_col}, FALSE)
            THEN 1
            ELSE 0
        END
    ) AS ga4_available_days,

    SUM(
        CASE
            WHEN {date_col} BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                 AND COALESCE({impressions_col}, 0) > 0
            THEN 1
            ELSE 0
        END
    ) AS active_days_first_half,

    -- March 16-31: future outcome only
    SUM(
        CASE
            WHEN {date_col} BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
            THEN COALESCE({impressions_col}, 0)
            ELSE 0
        END
    ) AS imp_second_half

FROM {FACT_MARCH}

WHERE {date_col} BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'

GROUP BY
    {client_col},
    {content_col}
"""

df = con.execute(query).df()

print("Number of pages:", len(df))
display(df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Number of pages: 331437


,client_id,content_id,imp_first_half,clicks_first_half,sessions_first_half,engaged_sessions_first_half,ga4_available_days,active_days_first_half,imp_second_half
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4173.0,6.0,0.0,0.0,0.0,15.0,2350.0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,245.0,0.0,0.0,0.0,0.0,15.0,208.0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,3705.0,3.0,0.0,0.0,0.0,15.0,1925.0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,2440.0,8.0,0.0,0.0,0.0,15.0,2504.0
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,14.0,0.0,0.0,0.0,0.0,9.0,28.0


In [6]:
position_features = con.execute(f"""
SELECT
    client_hash_id AS client_id,
    content_hash_id AS content_id,

    SUM(
        CASE
            WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                 AND gsc_data_available = TRUE
            THEN 1
            ELSE 0
        END
    ) AS gsc_available_days,

    SUM(
        CASE
            WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
            THEN COALESCE(gsc_sum_position, 0)
            ELSE 0
        END
    ) AS sum_position_first_half

FROM {FACT_MARCH}

WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

df = df.merge(
    position_features,
    on=["client_id", "content_id"],
    how="left"
)

df["avg_position_first_half"] = np.where(
    df["imp_first_half"] > 0,
    df["sum_position_first_half"] / df["imp_first_half"],
    np.nan
)

print("Rows:", len(df))

display(
    df[
        [
            "content_id",
            "imp_first_half",
            "clicks_first_half",
            "gsc_available_days",
            "avg_position_first_half"
        ]
    ].head(10)
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 331437


,content_id,imp_first_half,clicks_first_half,gsc_available_days,avg_position_first_half
0,content_7a105f548d9c6916,4173.0,6.0,15.0,6.265037
1,content_a3ea9792f793ec72,245.0,0.0,15.0,4.085714
2,content_36c36abc7650d7af,3705.0,3.0,15.0,6.297706
3,content_a7da352b73b02668,2440.0,8.0,15.0,7.370902
4,content_f39be42b42a4e8f6,14.0,0.0,9.0,10.285714
5,content_1855a661b4d36130,240.0,1.0,15.0,3.770833
6,content_5d412fba6e1a2582,131.0,0.0,15.0,9.809160
7,content_1f380a642aed423b,44.0,1.0,15.0,6.409091
8,content_22c063002b7c1caf,172.0,0.0,15.0,7.755814
9,content_aafb2ab7e5fc80d0,3104.0,16.0,15.0,5.542848


In [7]:
second_half_coverage = con.execute(f"""
SELECT
    client_hash_id AS client_id,
    content_hash_id AS content_id,

    SUM(
        CASE
            WHEN report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
                 AND gsc_data_available = TRUE
            THEN 1
            ELSE 0
        END
    ) AS gsc_available_days_second_half

FROM {FACT_MARCH}

WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

df = df.merge(
    second_half_coverage,
    on=["client_id", "content_id"],
    how="left"
)

df["gsc_available_days_second_half"] = (
    df["gsc_available_days_second_half"]
    .fillna(0)
)

print("Second-half coverage added.")

Second-half coverage added.


In [8]:
clean_df = df[
    (df["gsc_available_days"] == 15) &
    (df["gsc_available_days_second_half"] == 16) &
    (df["imp_first_half"] > 0)
].copy()

print("All pages:", len(df))
print("Clean pages:", len(clean_df))

print(
    "Percentage kept:",
    f"{len(clean_df) / len(df):.1%}"
)

All pages: 331437
Clean pages: 61796
Percentage kept: 18.6%


In [9]:
# Average daily impressions in each time window
clean_df["avg_daily_imp_first_half"] = (
    clean_df["imp_first_half"] / 15
)

clean_df["avg_daily_imp_second_half"] = (
    clean_df["imp_second_half"] / 16
)

# Percentage change from first half to second half
clean_df["impression_change_pct"] = (
    (
        clean_df["avg_daily_imp_second_half"]
        - clean_df["avg_daily_imp_first_half"]
    )
    / clean_df["avg_daily_imp_first_half"]
) * 100

# Same Week 4 target
# 1 = impressions declined by more than 20%
# 0 = did not decline by more than 20%
clean_df["is_declining_proxy"] = (
    clean_df["impression_change_pct"] < -20
).astype(int)

# CTR from March 1-15 only
clean_df["ctr_first_half"] = (
    clean_df["clicks_first_half"]
    / clean_df["imp_first_half"]
) * 100

print("Clean pages:", len(clean_df))
print("Declining pages:", clean_df["is_declining_proxy"].sum())

print(
    "Declining rate:",
    f"{clean_df['is_declining_proxy'].mean():.1%}"
)

Clean pages: 61796
Declining pages: 20023
Declining rate: 32.4%


In [10]:
n_clients = clean_df["client_id"].nunique()

print("Unique clients:", n_clients)
print("Eligible pages:", len(clean_df))

Unique clients: 34
Eligible pages: 61796


In [11]:
pages_per_client = (
    clean_df
    .groupby("client_id")
    .size()
)

print("Pages per client:")
print(pages_per_client.describe())

Pages per client:
count       34.000000
mean      1817.529412
std       3886.676279
min          1.000000
25%          9.000000
50%        295.000000
75%       1114.000000
max      16649.000000
dtype: float64


In [12]:
client_summary = (
    clean_df
    .groupby("client_id")
    .agg(
        pages=("content_id", "size"),
        declining_pages=("is_declining_proxy", "sum"),
        decline_rate=("is_declining_proxy", "mean")
    )
    .sort_values("pages", ascending=False)
)

client_summary["decline_rate"] *= 100

display(client_summary)

print("Total clients:", len(client_summary))

,pages,declining_pages,decline_rate
client_id,,,
client_73cda7b4e4f265ea,16649,5543,33.293291
client_62f4a7e64f5e0096,12768,4747,37.178885
client_23a62021009f63c4,9885,4150,41.982802
client_fef1a8f436438636,6373,1200,18.829437
client_08a6a72ff48e62c0,3956,433,10.945399
client_20259bd6705d81d4,2627,747,28.435478
client_e5c2aa26a8598242,1994,630,31.594784
client_3f0ce4d44fe94f3d,1207,370,30.654515
client_a80fca3f171ed1de,1163,349,30.008598


Total clients: 34


In [15]:
model_df = clean_df[
    clean_df["avg_position_first_half"].notna()
    & (clean_df["avg_position_first_half"] > 0)
].copy()

model_df = model_df.reset_index(drop=True)

print("Modeling pages:", len(model_df))
print("Clients:", model_df["client_id"].nunique())

Modeling pages: 61795
Clients: 34


In [16]:
RANDOM_STATE = 42
TEST_CLIENT_FRACTION = 0.20

clients = model_df["client_id"].drop_duplicates().to_numpy()

rng = np.random.default_rng(RANDOM_STATE)
rng.shuffle(clients)

n_test_clients = max(
    1,
    int(round(len(clients) * TEST_CLIENT_FRACTION))
)

test_clients = set(clients[:n_test_clients])
train_clients = set(clients[n_test_clients:])

train_df = model_df[
    model_df["client_id"].isin(train_clients)
].copy()

test_df = model_df[
    model_df["client_id"].isin(test_clients)
].copy()

print("Train clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

print()
print("Train pages:", len(train_df))
print("Test pages:", len(test_df))

print()
print(
    "Train decline rate:",
    f"{train_df['is_declining_proxy'].mean():.1%}"
)

print(
    "Test decline rate:",
    f"{test_df['is_declining_proxy'].mean():.1%}"
)

Train clients: 27
Test clients: 7

Train pages: 48907
Test pages: 12888

Train decline rate: 31.1%
Test decline rate: 37.2%


In [17]:
client_overlap = (
    set(train_df["client_id"])
    & set(test_df["client_id"])
)

print("Client overlap:", len(client_overlap))

assert len(client_overlap) == 0
assert len(train_df) + len(test_df) == len(model_df)

print("Split check passed.")

Client overlap: 0
Split check passed.


##Features


In [18]:
position_bins = [0, 3, 10, 20, 50, np.inf]

position_labels = [
    "Top 3",
    "4-10",
    "11-20",
    "21-50",
    "50+"
]

train_df["position_bucket"] = pd.cut(
    train_df["avg_position_first_half"],
    bins=position_bins,
    labels=position_labels,
    include_lowest=True
)

test_df["position_bucket"] = pd.cut(
    test_df["avg_position_first_half"],
    bins=position_bins,
    labels=position_labels,
    include_lowest=True
)

print("Train position buckets:")
display(
    train_df["position_bucket"]
    .value_counts()
    .sort_index()
)

print("\nTest position buckets:")
display(
    test_df["position_bucket"]
    .value_counts()
    .sort_index()
)

Train position buckets:


,count
position_bucket,
Top 3,4745
4-10,22713
11-20,9596
21-50,10668
50+,1185



Test position buckets:


,count
position_bucket,
Top 3,2808
4-10,7876
11-20,1297
21-50,712
50+,195


In [19]:
train_ctr_medians = (
    train_df
    .groupby(
        "position_bucket",
        observed=True
    )["ctr_first_half"]
    .median()
)

print("CTR medians learned from Train only:")
display(train_ctr_medians)

CTR medians learned from Train only:


,ctr_first_half
position_bucket,
Top 3,0.251792
4-10,0.204082
11-20,0.085935
21-50,0.018761
50+,0.000000


In [20]:
# Map the Train-derived CTR median to each page
train_df["position_ctr_median"] = (
    train_df["position_bucket"]
    .map(train_ctr_medians)
    .astype(float)
)

test_df["position_ctr_median"] = (
    test_df["position_bucket"]
    .map(train_ctr_medians)
    .astype(float)
)

# Reproduce the Week 4 low-CTR logic,
# but learn the threshold from Train clients only.
train_df["low_ctr_for_position"] = np.where(
    train_df["position_ctr_median"] > 0,
    (
        train_df["ctr_first_half"]
        < train_df["position_ctr_median"]
    ).astype(int),
    (
        train_df["ctr_first_half"] == 0
    ).astype(int)
)

test_df["low_ctr_for_position"] = np.where(
    test_df["position_ctr_median"] > 0,
    (
        test_df["ctr_first_half"]
        < test_df["position_ctr_median"]
    ).astype(int),
    (
        test_df["ctr_first_half"] == 0
    ).astype(int)
)

print(
    "Train low-CTR rate:",
    f"{train_df['low_ctr_for_position'].mean():.1%}"
)

print(
    "Test low-CTR rate:",
    f"{test_df['low_ctr_for_position'].mean():.1%}"
)

print(
    "Missing Train thresholds:",
    train_df["position_ctr_median"].isna().sum()
)

print(
    "Missing Test thresholds:",
    test_df["position_ctr_median"].isna().sum()
)

Train low-CTR rate: 51.0%
Test low-CTR rate: 61.8%
Missing Train thresholds: 0
Missing Test thresholds: 0


In [21]:
signal_check = (
    test_df
    .groupby("low_ctr_for_position")
    .agg(
        pages=("content_id", "size"),
        decline_rate=("is_declining_proxy", "mean")
    )
    .reset_index()
)

signal_check["decline_rate"] *= 100

display(signal_check)

,low_ctr_for_position,pages,decline_rate
0,0,4925,30.152284
1,1,7963,41.630039


In [22]:
feature_cols = [
    "imp_first_half",
    "clicks_first_half",
    "ctr_first_half",
    "avg_position_first_half",
    "active_days_first_half",
    "low_ctr_for_position",
]

target_col = "is_declining_proxy"

X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

y_train = train_df[target_col].copy()
y_test = test_df[target_col].copy()

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nFeatures:")
for feature in feature_cols:
    print("-", feature)

print("\nMissing values:")
print("Train:", X_train.isna().sum().sum())
print("Test:", X_test.isna().sum().sum())

Train shape: (48907, 6)
Test shape: (12888, 6)

Features:
- imp_first_half
- clicks_first_half
- ctr_first_half
- avg_position_first_half
- active_days_first_half
- low_ctr_for_position

Missing values:
Train: 0
Test: 0


In [23]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)


def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))

    top_k_idx = np.argsort(scores)[::-1][:k]

    return y_true[top_k_idx].mean()


def evaluate_model(name, y_true, probabilities, predictions):
    return {
        "model": name,

        "accuracy": accuracy_score(
            y_true,
            predictions
        ),

        "precision": precision_score(
            y_true,
            predictions,
            zero_division=0
        ),

        "recall": recall_score(
            y_true,
            predictions,
            zero_division=0
        ),

        "f1": f1_score(
            y_true,
            predictions,
            zero_division=0
        ),

        "precision_at_10": precision_at_k(
            y_true,
            probabilities,
            10
        ),

        "precision_at_100": precision_at_k(
            y_true,
            probabilities,
            100
        ),

        "average_precision": average_precision_score(
            y_true,
            probabilities
        ),

        "roc_auc": roc_auc_score(
            y_true,
            probabilities
        ),
    }

##Model — Logistic Regression

In [24]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


logistic_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        )
    )
])

logistic_model.fit(
    X_train,
    y_train
)

print("Logistic Regression trained.")

Logistic Regression trained.


In [26]:
logistic_prob = logistic_model.predict_proba(
    X_test
)[:, 1]

logistic_pred = logistic_model.predict(
    X_test
)

logistic_results = evaluate_model(
    name="Logistic Regression",
    y_true=y_test,
    probabilities=logistic_prob,
    predictions=logistic_pred
)

display(
    pd.DataFrame([logistic_results])
)

,model,accuracy,precision,recall,f1,precision_at_10,precision_at_100,average_precision,roc_auc
0,Logistic Regression,0.564091,0.42877,0.512917,0.467084,0.5,0.55,0.417256,0.575163


In [27]:
print(
    "Test base decline rate:",
    f"{y_test.mean():.1%}"
)

print(
    "Logistic Precision@100:",
    f"{logistic_results['precision_at_100']:.1%}"
)

print(
    "Logistic Precision@10:",
    f"{logistic_results['precision_at_10']:.1%}"
)

Test base decline rate: 37.2%
Logistic Precision@100: 55.0%
Logistic Precision@10: 50.0%


## test client on the frozen baseline

In [28]:
baseline_eval_df = model_df.copy()

# Same Week 4 position buckets
baseline_eval_df["position_bucket"] = pd.cut(
    baseline_eval_df["avg_position_first_half"],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=["Top 3", "4-10", "11-20", "21-50", "50+"],
    include_lowest=True
)

# Same Week 4 CTR medians:
# calculated across the original eligible baseline data
baseline_ctr_medians = (
    baseline_eval_df
    .groupby(
        "position_bucket",
        observed=True
    )["ctr_first_half"]
    .median()
)

baseline_eval_df["position_ctr_median"] = (
    baseline_eval_df["position_bucket"]
    .map(baseline_ctr_medians)
    .astype(float)
)

# Same Week 4 Low CTR rule
baseline_eval_df["low_ctr_for_position"] = np.where(
    baseline_eval_df["position_ctr_median"] > 0,
    baseline_eval_df["ctr_first_half"]
        < baseline_eval_df["position_ctr_median"],
    baseline_eval_df["ctr_first_half"] == 0
)

# Same Week 4 rule
baseline_eval_df["good_position"] = (
    baseline_eval_df["avg_position_first_half"] <= 20
)

baseline_eval_df["rule_match"] = (
    baseline_eval_df["good_position"]
    & baseline_eval_df["low_ctr_for_position"]
)

# Same Week 4 score
baseline_eval_df["baseline_score"] = np.where(
    baseline_eval_df["rule_match"],
    baseline_eval_df["imp_first_half"],
    0
)

# Evaluate ONLY on the same Test clients
baseline_test = baseline_eval_df[
    baseline_eval_df["client_id"].isin(test_clients)
].copy()

baseline_test = baseline_test.sort_values(
    [
        "baseline_score",
        "avg_position_first_half"
    ],
    ascending=[False, True]
)

baseline_p10 = (
    baseline_test
    .head(10)["is_declining_proxy"]
    .mean()
)

baseline_p100 = (
    baseline_test
    .head(100)["is_declining_proxy"]
    .mean()
)

print(
    "Test base rate:",
    f"{baseline_test['is_declining_proxy'].mean():.1%}"
)

print(
    "Baseline Precision@10:",
    f"{baseline_p10:.1%}"
)

print(
    "Baseline Precision@100:",
    f"{baseline_p100:.1%}"
)

print(
    "Logistic Precision@100:",
    f"{logistic_results['precision_at_100']:.1%}"
)

Test base rate: 37.2%
Baseline Precision@10: 50.0%
Baseline Precision@100: 51.0%
Logistic Precision@100: 55.0%


##Decision Tree

In [29]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(
    class_weight="balanced",
    max_depth=5,
    min_samples_leaf=50,
    random_state=42
)

tree_model.fit(
    X_train,
    y_train
)

print("Decision Tree trained.")

Decision Tree trained.


In [30]:
tree_prob = tree_model.predict_proba(
    X_test
)[:, 1]

tree_pred = tree_model.predict(
    X_test
)

tree_results = evaluate_model(
    name="Decision Tree",
    y_true=y_test,
    probabilities=tree_prob,
    predictions=tree_pred
)

display(
    pd.DataFrame([tree_results])
)

print(
    "Decision Tree Precision@100:",
    f"{tree_results['precision_at_100']:.1%}"
)

print(
    "Decision Tree Precision@10:",
    f"{tree_results['precision_at_10']:.1%}"
)

,model,accuracy,precision,recall,f1,precision_at_10,precision_at_100,average_precision,roc_auc
0,Decision Tree,0.593808,0.467562,0.653125,0.54498,0.2,0.38,0.456028,0.631648


Decision Tree Precision@100: 38.0%
Decision Tree Precision@10: 20.0%


In [32]:
current_comparison = pd.DataFrame([
    {
        "model": "Week 4 Baseline",
        "precision_at_10": baseline_p10,
        "precision_at_100": baseline_p100,
    },
    logistic_results,
    tree_results
])

display(
    current_comparison[
        [
            "model",
            "precision_at_10",
            "precision_at_100"
        ]
    ]
)

,model,precision_at_10,precision_at_100
0,Week 4 Baseline,0.5,0.51
1,Logistic Regression,0.5,0.55
2,Decision Tree,0.2,0.38


##Random Forest

In [33]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=25,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train,
    y_train
)

print("Random Forest trained.")

Random Forest trained.


In [34]:
rf_prob = rf_model.predict_proba(
    X_test
)[:, 1]

rf_pred = rf_model.predict(
    X_test
)

rf_results = evaluate_model(
    name="Random Forest",
    y_true=y_test,
    probabilities=rf_prob,
    predictions=rf_pred
)

display(
    pd.DataFrame([rf_results])
)

print(
    "Random Forest Precision@100:",
    f"{rf_results['precision_at_100']:.1%}"
)

print(
    "Random Forest Precision@10:",
    f"{rf_results['precision_at_10']:.1%}"
)

,model,accuracy,precision,recall,f1,precision_at_10,precision_at_100,average_precision,roc_auc
0,Random Forest,0.602033,0.475155,0.655417,0.550915,0.6,0.47,0.489476,0.653934


Random Forest Precision@100: 47.0%
Random Forest Precision@10: 60.0%


In [35]:
comparison_table = pd.DataFrame([
    {
        "model": "Week 4 Baseline",
        "precision_at_10": baseline_p10,
        "precision_at_100": baseline_p100,
        "average_precision": np.nan,
        "roc_auc": np.nan
    },
    logistic_results,
    tree_results,
    rf_results
])

display(
    comparison_table[
        [
            "model",
            "precision_at_10",
            "precision_at_100",
            "average_precision",
            "roc_auc"
        ]
    ].sort_values(
        "precision_at_100",
        ascending=False
    )
)

,model,precision_at_10,precision_at_100,average_precision,roc_auc
1,Logistic Regression,0.5,0.55,0.417256,0.575163
0,Week 4 Baseline,0.5,0.51,NaN,NaN
3,Random Forest,0.6,0.47,0.489476,0.653934
2,Decision Tree,0.2,0.38,0.456028,0.631648
